# Clase 032 — eval y query

**Parte 0** · VanderPlas cap. 3 § 3.13.

> 🎯 Filtros y cálculos como strings tipo SQL. Útil para legibilidad y datasets grandes.

> ⏱️ ~45 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import time
rng = np.random.default_rng(42)

## 1️⃣ `df.query` — filtro como string

In [ ]:
df = pd.DataFrame({
    'precio'    : rng.uniform(10, 1000, 20).round(2),
    'cantidad'  : rng.integers(1, 50, 20),
    'categoria' : rng.choice(['A', 'B', 'C'], 20),
})

# Filtro tradicional
mask = (df['precio'] > 100) & (df['cantidad'] < 30) & (df['categoria'] == 'A')
filtrado_a = df[mask]

# Equivalente con query
filtrado_b = df.query('precio > 100 and cantidad < 30 and categoria == "A"')

print('mismo resultado:', filtrado_a.equals(filtrado_b))
print(filtrado_b)

## 2️⃣ Variables locales con `@`

Referencia variables del scope con prefijo `@`:

In [ ]:
threshold_precio = 500
categoria_objetivo = 'A'

result = df.query('precio > @threshold_precio and categoria == @categoria_objetivo')
print(result)

## 3️⃣ `df.eval` — expresiones aritméticas

Calcula columnas sin temporales y, en datasets grandes, usando `numexpr` (más rápido).

In [ ]:
# Tradicional
df['total_a'] = df['precio'] * df['cantidad']

# Con eval
df['total_b'] = df.eval('precio * cantidad')

# inplace: añade al DataFrame
df.eval('descuento = precio * 0.1', inplace=True)

print(df[['precio', 'cantidad', 'total_a', 'total_b', 'descuento']].head())
print('\niguales total_a == total_b?', (df['total_a'] == df['total_b']).all())

## 4️⃣ Cuándo conviene query/eval

**Sí**:
- Cadenas de filtros largas → más legible una string que `(a) & (b) & (c) & (d)`.
- Datasets grandes (>10k filas) con expresiones complejas → `numexpr` da speedup.
- Filtros parametrizables (con `@`) sin construir máscaras complejas.

**No**:
- Datasets pequeños — overhead del parser no compensa.
- Cuando necesitas autocomplete del IDE — strings no se autocompletan.
- Cuando el filtro usa métodos custom (no es solo aritmética/comparación).

## 5️⃣ Benchmark — speedup en grandes

In [ ]:
N = 1_000_000
big = pd.DataFrame({
    'a': rng.normal(0, 1, N),
    'b': rng.normal(0, 1, N),
    'c': rng.choice(['x','y','z'], N),
})

t0 = time.perf_counter()
_ = big[(big['a'] > 0.5) & (big['b'] < -0.5) & (big['c'] == 'x')]
t1 = time.perf_counter()

t2 = time.perf_counter()
_ = big.query('a > 0.5 and b < -0.5 and c == "x"')
t3 = time.perf_counter()

print(f'tradicional : {(t1-t0)*1000:.1f} ms')
print(f'query       : {(t3-t2)*1000:.1f} ms')

## ✅ Checklist

- [ ] Sé escribir filtros largos con `df.query`
- [ ] Uso `@var` para referenciar variables locales
- [ ] Uso `df.eval` para columnas derivadas sin temporales
- [ ] Sé que el speedup aparece en datasets grandes
- [ ] Reconozco trade-off: legibilidad vs autocomplete IDE

## 📝 Homework

Ver `README.md`. 3 filtros equivalentes, eval para cols, benchmark en 1M filas.

## 📖 Definiciones y características

**`df.query()`**

Filtro como string tipo SQL: `df.query('precio > 100 and categoria == "A"')`. Más legible que máscara booleana compuesta cuando hay >2 condiciones.

**`df.eval()`**

Evalúa expresiones aritméticas: `df.eval('total = precio * cantidad')`. Con `inplace=True` añade la columna al DF.

**Variables locales (`@var`)**

Dentro de query/eval, prefijo `@` referencia variables del scope Python: `df.query('x > @threshold')`.

**`numexpr`**

Motor de cómputo que vectoriza expresiones en C/SIMD. Usado por eval/query bajo el capó cuando los datasets son grandes. Acelera operaciones aritméticas complejas.

**Trade-off legibilidad vs IDE**

Query strings: más legibles para humanos. Filtros tradicionales: autocomplete del IDE, type checking. Elige según contexto.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `UndefinedVariableError: name 'X' is not defined` | Variable Python no prefijada con `@`. **Fix**: `df.query('x > @threshold')` no `df.query('x > threshold')`. |
| Strings dentro de query con comillas dobles dan error | Mezcla de quotes. **Fix**: usa quotes opuestas: `df.query("categoria == 'A'")` o triple-quoted. |
| `df.eval('col_nueva = ...')` no añade la columna | Sin `inplace=True`. **Fix**: `df.eval('col = x*y', inplace=True)` o `df = df.eval('col = x*y')`. |
| query/eval más lento que máscara tradicional en mi caso | Para datasets pequeños (<10k filas), el overhead del parser no compensa. **Fix**: usa máscara tradicional ahí; reserva query/eval para datasets grandes. |
| Funciones custom no funcionan en query | Solo aritmética + operadores. **Fix**: `df.query('x > @threshold')` con cálculo previo, o usa máscara tradicional con la función. |

## ❓ Preguntas frecuentes

**❓ ¿`query` o máscara tradicional?**

**Máscara** para filtros simples (1-2 condiciones) y autocomplete del IDE. **`query`** para filtros largos (4+ condiciones), parametrizables (`@var`), o cuando el lector lo encuentra más claro.

**❓ ¿`eval` y `query` siempre son más rápidos?**

**No** — solo en datasets grandes (>100k) con expresiones complejas. Para pequeños son iguales o ligeramente más lentos.

**❓ ¿Qué operadores acepta query?**

Aritméticos (`+ - * / **`), comparación (`==`, `<`, `>`, `<=`, `>=`, `!=`), boolean (`and`, `or`, `not` o `&`, `|`, `~`), `in`, `not in`. No funciones.

**❓ ¿`query` reemplaza a SQL en pandas?**

Para filtros, sí (mismo nivel expresivo). Para joins y aggregations, no — usa `merge` y `groupby` clásicos. Para datasets >1GB, considera DuckDB directo.

**❓ ¿Hay `eval` peligroso (security)?**

`pd.eval` no usa `exec()` Python — es parser propio limitado. No ejecuta arbitrary code. **Pero**: nunca pases strings de usuario sin sanitizar a query/eval.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.13
- [pandas enhancing perf](https://pandas.pydata.org/docs/user_guide/enhancingperf.html)

➡️ **Siguiente:** [033 — Polars: DataFrames modernos](../033-polars-dataframes-modernos/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Filtro tradicional vs `query`.**

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(32)
df = pd.DataFrame({'a': rng.integers(0, 20, 1000), 'b': rng.integers(0, 10, 1000),
                   'c': rng.choice(['x', 'y'], 1000)})
trad = df[(df.a > 10) & (df.b < 5) & (df.c == 'x')]
q = df.query('a > 10 and b < 5 and c == "x"')
assert trad.equals(q)
print('filas:', len(q), '- query es mas legible y da el mismo resultado.')

**Ej. 2 — Variable local con `@`.**

In [ ]:
threshold = 12
r = df.query('a > @threshold')
assert (r['a'] > 12).all()
print('filas con a > threshold(@):', len(r))

**Ej. 3 — `eval` para una nueva columna** (`inplace=True`).

In [ ]:
df2 = pd.DataFrame({'precio': [10, 20, 30], 'cantidad': [2, 3, 1]})
df2.eval('total = precio * cantidad', inplace=True)
print(df2)
assert (df2['total'] == [20, 60, 30]).all()

**Ej. 4 — Benchmark** filtro tradicional vs `query` (1M filas).

In [ ]:
import timeit
big = pd.DataFrame({'a': rng.integers(0, 100, 1_000_000), 'b': rng.integers(0, 100, 1_000_000)})
t_trad = timeit.timeit(lambda: big[(big.a > 50) & (big.b < 50)], number=5)
t_q    = timeit.timeit(lambda: big.query('a > 50 and b < 50'), number=5)
print(f'tradicional: {t_trad*1000:.1f} ms | query: {t_q*1000:.1f} ms')
assert t_trad > 0 and t_q > 0

**Ej. 5 — `eval(inplace=False)`** vs cálculo tradicional (idénticos).

In [ ]:
total_eval = df2.eval('precio * cantidad')     # inplace=False -> Series
total_trad = df2['precio'] * df2['cantidad']
assert total_eval.equals(total_trad)
print('eval(inplace=False) y el calculo tradicional coinciden.')